In [ ]:
%pip install selenium
%pip install webdriver-manager
%pip install tqdm

## 모듈화한 함수 불러서 쓰기(각 링크 txt저장까지)

In [2]:
import sys
from pathlib import Path

# 현재 노트북 위치 (AC 폴더)
CURRENT_DIR = Path.cwd()

# 무조건 한 칸 위 (data_crawling)
BASE_DIR = CURRENT_DIR.parent

# import 경로 추가 (맨 앞에 넣는 게 더 안전)
sys.path.insert(0, str(BASE_DIR))

from selenium_auto_module import full_page, product_links


URL = "https://www.lge.co.kr/category/washing-machines"


html_sample = full_page(URL, "washing_machine")
print(html_sample)

links_sample = product_links("washing_machine")
print(links_sample)

페이지 로딩 완료
1번째 클릭
2번째 클릭
3번째 클릭
4번째 클릭
5번째 클릭
끝까지 펼침 완료
HTML 길이: 1095601
HTML 저장 완료: washing_machine_full_page_html.txt
<html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><link rel="preload" as="image" href="/kr/common/footer/sns_youtube.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_insta.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_facebook.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_newsroom.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_kakao.svg"><link rel="preload" as="image"
제품 카드 개수: 156
추출된 링크 개수: 156
저장 완료: washing_machine_product_links.txt
['https://www.lge.co.kr/washing-machines/f24wdwpr', 'https://www.lge.co.kr/washing-machines/f8vvr', 'https://www.lge.co.kr/washing-machines/f9wtbq', 'https://www.lge.co.kr/washing-machines/f9wpb', 'https://www.lge.co.kr/washing-machines/fg24v-20v-akor2']


## 재원추출 및 csv저장

In [2]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# 1. 드라이버 실행
# =========================
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 10)


# =========================
# 2. 세탁기 상품 링크 txt 읽기
# =========================
with open("washing_machine_product_links.txt", "r", encoding="utf-8") as f:
    product_links = [line.strip() for line in f if line.strip()]

print("총 링크 개수:", len(product_links))


# =========================
# 3. 공통 함수
# =========================
def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_product_name(soup):
    tag = soup.select_one("h1.name")
    if not tag:
        return ""

    for child in tag.select("span, em"):
        child.decompose()

    return clean_text(tag.get_text())


def get_image_url(soup):
    img = soup.select_one("#base_detail_target")
    if not img:
        return ""

    src = img.get("src", "").strip()
    if not src:
        return ""

    if src.startswith("http"):
        return src

    return "https://www.lge.co.kr" + src


def get_price(soup):
    tag = soup.select_one("small.original")
    if not tag:
        return ""

    for blind in tag.select(".blind"):
        blind.decompose()

    return clean_text(tag.get_text())


def get_spec(soup, *labels):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))

        for label in labels:
            if label in dt_text:
                dd = dt.find_next_sibling("dd")
                if dd:
                    return clean_text(dd.get_text(" ", strip=True))

    return ""


def get_manual_pdf_url(soup):
    for top_area in soup.select("div.top-area"):
        strong = top_area.select_one("strong")
        title = clean_text(strong.get_text()) if strong else ""

        # "제품 사용 설명서(바로보기용)" 제외
        if title == "제품 사용 설명서":
            parent = top_area.find_parent()

            if parent:
                a_tag = parent.select_one("div.btn-group a[href]")

                if a_tag:
                    href = a_tag.get("href", "").strip()

                    if href.startswith("http"):
                        return href
                    elif href.startswith("/"):
                        return "https://www.lge.co.kr" + href
                    else:
                        return href

    return ""


# =========================
# 4. 전체 링크 순회
# =========================
rows = []
failures = []

pbar = tqdm(product_links, desc="세탁기 상품 크롤링", ncols=120)

for idx, url in enumerate(pbar, start=1):
    try:
        pbar.set_postfix_str(f"{idx}/{len(product_links)} 접속 중")

        driver.get(url)
        time.sleep(3)

        # 제품 정보 / 제품 스펙 더 보기 클릭
        try:
            more_btn = wait.until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        "//a[contains(@title,'펼치기') or contains(., '제품 정보 더 보기') or contains(., '제품스펙 상세정보 펼치기')]"
                    )
                )
            )

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});",
                more_btn
            )
            time.sleep(1)

            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(2)

        except:
            pass

        soup = BeautifulSoup(driver.page_source, "html.parser")

        row = {
            "제품명": get_product_name(soup),
            "이미지 링크": get_image_url(soup),
            "가격(원)": get_price(soup),

            "에너지 소비효율등급": get_spec(
                soup,
                "에너지소비효율등급",
                "에너지 소비효율등급",
                "에너지소비효율 등급",
                "에너지 소비효율 등급"
            ),

            "소비전력(W)": get_spec(
                soup,
                "소비전력",
                "소비 전력"
            ),

            "세탁 용량(kg)": get_spec(
                soup,
                "세탁용량",
                "세탁 용량"
            ),

            "건조 용량(kg)": get_spec(
                soup,
                "건조용량",
                "건조 용량"
            ),

            "크기(mm)": get_spec(
                soup,
                "제품 크기",
                "제품크기",
                "크기 (WxHxD",
                "크기(WxHxD",
                "크기"
            ),

            "무게(kg)": get_spec(
                soup,
                "무게",
                "제품 무게",
                "제품무게"
            ),

            "색상": get_spec(
                soup,
                "색상",
                "컬러",
                "색깔"
            ),

            "도어 디자인": get_spec(
                soup,
                "도어 디자인",
                "도어디자인",
                "도어"
            ),

            "조작 및 표현방식": get_spec(
                soup,
                "조작 및 표현방식",
                "조작 및 표현 방식",
                "조작/표현방식",
                "조작부",
                "표현방식"
            ),

            "물온도": get_spec(
                soup,
                "물온도",
                "물 온도"
            ),

            "탈수선택": get_spec(
                soup,
                "탈수선택",
                "탈수 선택",
                "탈수"
            ),

            "제품 사용 설명서": get_manual_pdf_url(soup),
        }

        rows.append(row)

        pbar.set_postfix_str(f"완료: {row['제품명'][:25]}")

        if idx % 10 == 0:
            pd.DataFrame(rows).to_csv(
                "lge_wm_temp.csv",
                index=False,
                encoding="utf-8-sig"
            )

    except Exception as e:
        failures.append({"url": url, "error": str(e)})
        pbar.set_postfix_str(f"실패: {idx}/{len(product_links)}")


# =========================
# 5. 최종 CSV 저장
# =========================
df = pd.DataFrame(rows)

df.to_csv(
    "lge_wm_all_products.csv",
    index=False,
    encoding="utf-8-sig"
)

print("전체 저장 완료: lge_wm_all_products.csv")
print("총 수집 개수:", len(df))
print("실패 개수:", len(failures))

display(df.head())


# =========================
# 6. 실패 목록 저장
# =========================
if failures:
    pd.DataFrame(failures).to_csv(
        "lge_wm_failures.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print("실패 목록 저장 완료: lge_wm_failures.csv")


# =========================
# 7. 드라이버 종료
# =========================
driver.quit()

총 링크 개수: 156


세탁기 상품 크롤링: 100%|█████████████████████████████████████| 156/156 [30:00<00:00, 11.54s/it, 완료: LG 통돌이 세탁기]접속 중]/s]

전체 저장 완료: lge_wm_all_products.csv
총 수집 개수: 156
실패 개수: 0


,제품명,이미지 링크,가격(원),에너지 소비효율등급,소비전력(W),세탁 용량(kg),건조 용량(kg),크기(mm),무게(kg),색상,도어 디자인,조작 및 표현방식,물온도,탈수선택,제품 사용 설명서
0,LG 트롬 세탁기,https://www.lge.co.kr/kr/images/washing-machin...,"1,150,000원",1등급,"2,000",12,,600 x 850 x 615,73.0,모던 스테인리스,"600 x 850 x 1,145",다이얼+푸시버튼 및 LED 디스플레이,"냉수, 30, 40, 60, 95℃",O 있음,https://gscs-b2c.lge.com/open/downloadFile?fil...
1,LG 트롬 세탁기,https://www.lge.co.kr/kr/images/washing-machin...,"1,050,000원",1등급,"2,000",12,,600 x 850 x 615,73.0,릴리 화이트,"600 x 850 x 1,145",다이얼+푸시버튼 및 LED 디스플레이,"냉수, 30, 40, 60, 95℃",O 있음,https://gscs-b2c.lge.com/open/downloadFile?fil...
2,LG 트롬 세탁기,https://www.lge.co.kr/kr/images/washing-machin...,"1,250,000원",1등급,"2,000",15,,645 x 940 x 770,78.0,스테인리스 블랙,"645x940x1,280",다이얼 + LED 터치 디스플레이,"냉수, 30, 40, 60, 95℃",O 있음,https://gscs-b2c.lge.com/open/downloadFile?fil...
3,LG 트롬 세탁기,https://www.lge.co.kr/kr/images/washing-machin...,"1,250,000원",1등급,"2,000",17,,650x950x780,80.0,글로우 에센스 화이트(유광),"650x950x1,280",다이얼+풀터치버튼 및 LED 디스플레이,"냉수, 20, 40, 60, 95℃",O 있음,
4,LG 트롬 세탁기,https://www.lge.co.kr/kr/images/washing-machin...,"1,200,000원",3등급,"2,000",17,,650x950x780,80.0,에센스 그라파이트(무광),"650x950x1,280",다이얼+풀터치버튼 및 LED 디스플레이,"냉수, 20, 40, 60, 95℃",O 있음,https://gscs-b2c.lge.com/open/downloadFile?fil...
